**Deliverable A - Python program**

Cell 1: Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
from google.colab import files

# 1. Interactive file upload
print("Please upload the dataset CSV file:")
uploaded = files.upload()

# Get the filename dynamically (in case it was renamed)
file_name = list(uploaded.keys())[0]

# Load the dataset into Pandas
df = pd.read_csv(io.BytesIO(uploaded[file_name]))

# Verify successful loading: First and last 5 rows
display(df.head())
display(df.tail())

# Dataset Overview
print("\n--- Dataset Info ---")
df.info()

print("\n--- Missing Values ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

Cell 2: Attribute Selection and Data Cleaning

In [ ]:
# 2. Attribute Selection
# Drop 'company' due to massive missing values (~94% missing)
# Drop 'agent' as it represents IDs and contains significant missing values
# Drop 'reservation_status_date' as it is redundant for general predictive EDA
df_clean = df.drop(columns=['company', 'agent', 'reservation_status_date'])

# Handle minor missing values appropriately
df_clean['country'] = df_clean['country'].fillna(df_clean['country'].mode()[0])
df_clean['children'] = df_clean['children'].fillna(0)

print(f"Data shape after cleaning: {df_clean.shape}")

Cell 3: Exploratory Data Analysis (EDA)

In [ ]:
# 3. Exploratory Data Analysis (EDA)
# Categorical summaries
print("--- Categorical Summaries ---")
print("Hotel Type Proportion:\n", df_clean['hotel'].value_counts(normalize=True))
print("\nCancellation Rate:\n", df_clean['is_canceled'].value_counts(normalize=True))

# Numerical summaries
print("\n--- Numerical Summaries ---")
display(df_clean[['lead_time', 'adr', 'stays_in_week_nights', 'total_of_special_requests']].describe())

# Visualization: Cancellations by Hotel Type
plt.figure(figsize=(8, 5))
sns.countplot(data=df_clean, x='hotel', hue='is_canceled', palette='Set2')
plt.title('Hotel Cancellations by Hotel Type')
plt.xlabel('Hotel Type')
plt.ylabel('Count')
plt.show()

Cell 4: Global Correlation Matrix

In [ ]:
# 4. Correlation Matrix
# Select key numerical attributes for correlation
num_vars = ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights',
            'adults', 'children', 'babies', 'previous_cancellations',
            'booking_changes', 'days_in_waiting_list', 'adr',
            'required_car_parking_spaces', 'total_of_special_requests', 'is_canceled']

corr_matrix = df_clean[num_vars].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Overall Correlation Matrix of Numerical Variables')
plt.tight_layout()
plt.show()

# Display top correlations with target variable 'is_canceled'
print("Top Positive Correlations with 'is_canceled':")
print(corr_matrix['is_canceled'].sort_values(ascending=False).head(4)[1:])
print("\nTop Negative Correlations with 'is_canceled':")
print(corr_matrix['is_canceled'].sort_values(ascending=True).head(3))

Cell 5: Group-Based Correlation Analysis

In [ ]:
# 5. Group-Based Correlation (Segmented by Hotel Type)
resort_corr = df_clean[df_clean['hotel'] == 'Resort Hotel'][num_vars].corr()
city_corr = df_clean[df_clean['hotel'] == 'City Hotel'][num_vars].corr()

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Resort Hotel Matrix
sns.heatmap(resort_corr, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, ax=axes[0], cbar=False)
axes[0].set_title('Correlation Matrix: Resort Hotel')

# City Hotel Matrix
sns.heatmap(city_corr, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title('Correlation Matrix: City Hotel')

plt.tight_layout()
plt.show()